In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns",None)

In [2]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
)

CURATED_DIR = (
    PROJECT_DIR
    / "data"
    / "curated"
)

In [4]:
policy_360 = pd.read_csv(
    CURATED_DIR / "policy_360.csv",
    low_memory=False
)

renewals_df = pd.read_csv(
    RAW_DIR / "renewals.csv"
)

premiums_df = pd.read_csv(
    RAW_DIR / "premiums.csv"
)

claims_df = pd.read_csv(
    RAW_DIR / "claims.csv"
)

In [5]:
print("Policy 360:", policy_360.shape)
print("Renewals  :", renewals_df.shape)
print("Premiums  :", premiums_df.shape)
print("Claims    :", claims_df.shape)

Policy 360: (150000, 88)
Renewals  : (20000, 12)
Premiums  : (100000, 10)
Claims    : (75000, 14)


In [6]:
renewals_df["RENEWAL_DUE_DATE"] = pd.to_datetime(renewals_df["RENEWAL_DUE_DATE"], errors="coerce")

In [7]:
premiums_df["DUE_DATE"] = pd.to_datetime(premiums_df["DUE_DATE"], errors="coerce")

premiums_df["PAID_DATE"] = pd.to_datetime(premiums_df["PAID_DATE"], errors="coerce")

Keep Only Finalized Renewals
Renewed
Lapsed
Cancelled
Pending

In [8]:
finalized_df = renewals_df[renewals_df["RENEWAL_STATUS"] != "Pending"].copy()

In [9]:
finalized_df.shape

(19204, 12)

In [10]:
finalized_df["RENEWED_FLAG"] = (finalized_df["RENEWAL_STATUS"] == "Renewed").astype(int)

Renewed = 1

Lapsed / Cancelled = 0

But for retention modelling, our real business interest is:
Who is at risk of NOT renewing?

In [11]:
finalized_df["NON_RENEWAL_FLAG"] = 1 - finalized_df["RENEWED_FLAG"]

I prefer this as the ML target because:
1 now means the customer we want the retention team to identify.

In [12]:
finalized_df["NON_RENEWAL_FLAG"].value_counts()

NON_RENEWAL_FLAG
0    17334
1     1870
Name: count, dtype: int64

In [13]:
finalized_df["NON_RENEWAL_FLAG"].value_counts(normalize=True).mul(100).round(2)

NON_RENEWAL_FLAG
0    90.26
1     9.74
Name: proportion, dtype: float64

Class Imbalance
Our minority class is:
Non-Renewed ≈ 9.7%

Build Renewal Base Table
Take only relevant renewal information:

In [14]:
renewal_base = finalized_df[
    [
        "POLICY_ID",
        "CUSTOMER_ID",
        "RENEWAL_DUE_DATE",
        "OLD_PREMIUM",
        "NEW_PREMIUM",
        "PREMIUM_INCREASE_PCT",
        "LOYALTY_YEARS",
        "CONTACT_CHANNEL",
        "RENEWED_FLAG",
        "NON_RENEWAL_FLAG"
    ]
].copy()

excluded-->RENEWAL_PROBABILITY
Potential leakage.
CLAIM_HISTORY_FLAG
We found it doesn't agree reliably with the actual Claims dataset.
We'll calculate claim history ourselves.
RENEWAL_STATUS
Direct target information.

In [15]:
static_cols = [
    "POLICY_ID",
    "CUSTOMER_ID",

    # Customer
    "AGE",
    "GENDER",
    "MARITAL_STATUS",
    "OCCUPATION",
    "ANNUAL_INCOME",
    "STATE",
    "CREDIT_SCORE",
    "CUSTOMER_RISK_SEGMENT",

    # Policy
    "POLICY_TYPE",
    "SALES_CHANNEL",
    "SUM_INSURED",
    "ANNUAL_PREMIUM",
    "PAYMENT_MODE",
    "RISK_SCORE",
    "RISK_BAND",
    "DISCOUNT_PCT",
    "COMMISSION_PCT",
    "POLICY_TERM_DAYS",
    "PREMIUM_TO_SUM_INSURED_PCT",

    # Underwriting
    "HEALTH_SCORE",
    "LIFESTYLE",
    "MEDICAL_HISTORY_FLAG",
    "SMOKER_FLAG",
    "BMI",
    "OCCUPATION_RISK",
    "UNDERWRITING_SCORE",
    "UW_DECISION",
    "PREMIUM_LOADING_PCT",
    "HAS_UNDERWRITING_RECORD"
]

In [16]:
static_features = ( policy_360[  static_cols  ].copy())

Why Exclude POLICY_STATUS?
This is subtle.
You might think:
POLICY_STATUS

would be a good predictor.
But suppose after renewal:
Lapsed customer
→ POLICY_STATUS = Lapsed

Then the model could simply learn:
Lapsed status
→ customer did not renew

That's target leakage.
Therefore, for our first clean model:
Do not use POLICY_STATUS.

In [17]:
model_df = renewal_base.merge(static_features, on=["POLICY_ID", "CUSTOMER_ID"], how="left", validate="one_to_one")

In [18]:
print(model_df.shape)

print(model_df[ "POLICY_ID"].duplicated().sum())

(19204, 39)
0


In [19]:
model_df["PREMIUM_CHANGE_AMOUNT"] = model_df["NEW_PREMIUM"] - model_df["OLD_PREMIUM"]

In [20]:
model_df["PREMIUM_INCREASED_FLAG"] = (model_df["NEW_PREMIUM"] > model_df["OLD_PREMIUM"]).astype(int)

In [21]:
model_df["HIGH_PREMIUM_INCREASE_FLAG"] = (model_df["PREMIUM_INCREASE_PCT"] >= 15).astype(int)

Premium Affordability Feature

In [22]:
model_df["NEW_PREMIUM_TO_INCOME_PCT"] = (model_df["NEW_PREMIUM"] / model_df["ANNUAL_INCOME"]) * 100

how much income goes in premium -->Premium-to-income ratio proxy

Time-Aware Payments-->
Earlier our Policy_360 payment features were calculated from the entire premium table.
For ML, we need:
Only payment behavior available before renewal.



Attach Renewal Date to Payment Transactions

In [23]:
payment_history = premiums_df.merge(
    finalized_df[["POLICY_ID", "CUSTOMER_ID", "RENEWAL_DUE_DATE"]],
    on=["POLICY_ID", "CUSTOMER_ID"],
    how="inner",
    validate="many_to_one"
)

Keep Transactions Due Before Renewal

In [24]:
payment_history = payment_history[payment_history["DUE_DATE"] < payment_history["RENEWAL_DUE_DATE"]].copy()

Was Payment Completed Before Renewal?

In [27]:
payment_history["PAID_BEFORE_RENEWAL"] = (payment_history["PAID_DATE"].notna() & (payment_history["PAID_DATE"] <= payment_history["RENEWAL_DUE_DATE"])).astype(int)

Was It Still Unpaid at Renewal?

In [28]:
payment_history["UNPAID_AT_RENEWAL"] = (payment_history["PAID_DATE"].isna() | (payment_history["PAID_DATE"] > payment_history["RENEWAL_DUE_DATE"])).astype(int)

UNPAID_AT_RENEWAL = 1
because the insurer didn't yet have that payment.

Calculate Historical Payment Delay

In [29]:
payment_history["PAYMENT_DELAY_PRIOR"] = np.where(
    payment_history["PAID_BEFORE_RENEWAL"] == 1,
    (payment_history["PAID_DATE"] - payment_history["DUE_DATE"]).dt.days,
    np.nan
)

Calculate Historical Paid Amount

In [30]:
payment_history["PAID_AMOUNT_PRIOR"] = np.where(
    payment_history["PAID_BEFORE_RENEWAL"] == 1,
    payment_history["PREMIUM_AMOUNT"],
    0
)

Aggregate Time-Aware Payment Features

In [31]:
payment_ml_features = payment_history.groupby(
    ["POLICY_ID", "CUSTOMER_ID"]
).agg(
    TOTAL_DUE_TXNS_PRIOR=("PREMIUM_TXN_ID", "count"),
    PAID_TXNS_PRIOR=("PAID_BEFORE_RENEWAL", "sum"),
    UNPAID_TXNS_AT_RENEWAL=("UNPAID_AT_RENEWAL", "sum"),
    TOTAL_PREMIUM_DUE_PRIOR=("PREMIUM_AMOUNT", "sum"),
    TOTAL_PREMIUM_PAID_PRIOR=("PAID_AMOUNT_PRIOR", "sum"),
    AVG_PAYMENT_DELAY_PRIOR=("PAYMENT_DELAY_PRIOR", "mean"),
    MAX_PAYMENT_DELAY_PRIOR=("PAYMENT_DELAY_PRIOR", "max")
).reset_index()

Create Historical Payment Success Rate

In [32]:
payment_ml_features["PAYMENT_SUCCESS_RATE_PRIOR"] = payment_ml_features["PAID_TXNS_PRIOR"] / payment_ml_features["TOTAL_DUE_TXNS_PRIOR"]

In [33]:
payment_ml_features[
    "HAS_PAYMENT_HISTORY_PRIOR"
] = 1

In [34]:
payment_ml_features.head()

,POLICY_ID,CUSTOMER_ID,TOTAL_DUE_TXNS_PRIOR,PAID_TXNS_PRIOR,UNPAID_TXNS_AT_RENEWAL,TOTAL_PREMIUM_DUE_PRIOR,TOTAL_PREMIUM_PAID_PRIOR,AVG_PAYMENT_DELAY_PRIOR,MAX_PAYMENT_DELAY_PRIOR,PAYMENT_SUCCESS_RATE_PRIOR,HAS_PAYMENT_HISTORY_PRIOR
0,POL000000041,CUST00041169,1,1,0,24716,24716,2.000000,2.0,1.0,1
1,POL000000055,CUST00044166,1,1,0,8946,8946,20.000000,20.0,1.0,1
2,POL000000057,CUST00022002,1,1,0,1495,1495,14.000000,14.0,1.0,1
3,POL000000071,CUST00088904,3,3,0,69828,69828,10.333333,27.0,1.0,1
4,POL000000082,CUST00056804,1,1,0,1795,1795,3.000000,3.0,1.0,1


Merge Payment Features

In [35]:
model_df = model_df.merge(
    payment_ml_features,
    on=["POLICY_ID", "CUSTOMER_ID"],
    how="left",
    validate="one_to_one"
)

In [36]:
# Handle Missing Payment History
model_df["HAS_PAYMENT_HISTORY_PRIOR"] = model_df["HAS_PAYMENT_HISTORY_PRIOR"].fillna(0).astype(int)

In [37]:
payment_zero_cols = [
    "TOTAL_DUE_TXNS_PRIOR",
    "PAID_TXNS_PRIOR",
    "UNPAID_TXNS_AT_RENEWAL",
    "TOTAL_PREMIUM_DUE_PRIOR",
    "TOTAL_PREMIUM_PAID_PRIOR"
]

model_df[payment_zero_cols] = model_df[payment_zero_cols].fillna(0)

Now Create Time-Aware Claim Features

In [38]:
claim_history = claims_df.merge(
    finalized_df[["POLICY_ID", "CUSTOMER_ID", "RENEWAL_DUE_DATE"]],
    on=["POLICY_ID", "CUSTOMER_ID"],
    how="inner",
    validate="many_to_one"
)

Keep Claims Known Before Renewal

In [39]:
claim_history = claim_history[claim_history["REPORTED_DATE"] <= claim_history["RENEWAL_DUE_DATE"]].copy()

In [40]:
# Aggregate Historical Claims
claim_ml_features = claim_history.groupby(
    ["POLICY_ID", "CUSTOMER_ID"]
).agg(
    TOTAL_CLAIMS_PRIOR=("CLAIM_ID", "count"),
    TOTAL_CLAIM_AMOUNT_PRIOR=("CLAIM_AMOUNT", "sum"),
    AVG_CLAIM_AMOUNT_PRIOR=("CLAIM_AMOUNT", "mean"),
    MAX_CLAIM_AMOUNT_PRIOR=("CLAIM_AMOUNT", "max"),
    LAST_CLAIM_REPORTED_DATE=("REPORTED_DATE", "max")
).reset_index()

In [41]:
claim_ml_features[
    "HAS_CLAIM_HISTORY_PRIOR"
] = 1

In [43]:
model_df = model_df.merge(
    claim_ml_features,
    on=[
        "POLICY_ID",
        "CUSTOMER_ID"
    ],
    how="left",
    validate="one_to_one"
)

In [44]:
model_df[
    "HAS_CLAIM_HISTORY_PRIOR"
] = (
    model_df[
        "HAS_CLAIM_HISTORY_PRIOR"
    ]
    .fillna(0)
    .astype(int)
)

In [45]:
claim_zero_cols = [
    "TOTAL_CLAIMS_PRIOR",
    "TOTAL_CLAIM_AMOUNT_PRIOR"
]

model_df[
    claim_zero_cols
] = (
    model_df[
        claim_zero_cols
    ]
    .fillna(0)
)

Create Claim Recency

In [50]:
model_df["RENEWAL_DUE_DATE"] = pd.to_datetime(model_df["RENEWAL_DUE_DATE"], errors="coerce")

model_df["LAST_CLAIM_REPORTED_DATE"] = pd.to_datetime(model_df["LAST_CLAIM_REPORTED_DATE"], errors="coerce")

In [51]:
model_df["DAYS_SINCE_LAST_CLAIM"] = (
    model_df["RENEWAL_DUE_DATE"] - model_df["LAST_CLAIM_REPORTED_DATE"]
).dt.days

In [52]:
model_df["HAS_UNPAID_PAYMENT_AT_RENEWAL"] = (model_df["UNPAID_TXNS_AT_RENEWAL"] > 0).astype(int)

Check Our Model Dataset

In [53]:
print(model_df.shape)
print(model_df["POLICY_ID"].nunique())
print(model_df["POLICY_ID"].duplicated().sum())

(19204, 60)
19204
0


Our grain is now:
1 row = 1 finalized renewal opportunity

This is different from Policy 360's grain conceptually.
Policy 360:
1 row = 1 policy

ML table:
1 row = 1 policy renewal opportunity

Identify Columns We Will Never Feed to the Model

In [54]:
id_cols = [
    "POLICY_ID",
    "CUSTOMER_ID",
    "RENEWAL_DUE_DATE"
]

In [55]:
target_cols = [
    "RENEWED_FLAG",
    "NON_RENEWAL_FLAG"
]

Numeric Features

In [58]:
numeric_features = [
    "AGE",
    "ANNUAL_INCOME",
    "CREDIT_SCORE",

    "SUM_INSURED",
    "ANNUAL_PREMIUM",
    "RISK_SCORE",
    "DISCOUNT_PCT",
    "COMMISSION_PCT",
    "POLICY_TERM_DAYS",
    "PREMIUM_TO_SUM_INSURED_PCT",

    "HEALTH_SCORE",
    "BMI",
    "UNDERWRITING_SCORE",
    "PREMIUM_LOADING_PCT",

    "OLD_PREMIUM",
    "NEW_PREMIUM",
    "PREMIUM_INCREASE_PCT",
    "PREMIUM_CHANGE_AMOUNT",
    "NEW_PREMIUM_TO_INCOME_PCT",
    "LOYALTY_YEARS",

    "TOTAL_DUE_TXNS_PRIOR",
    "PAID_TXNS_PRIOR",
    "UNPAID_TXNS_AT_RENEWAL",
    "TOTAL_PREMIUM_DUE_PRIOR",
    "TOTAL_PREMIUM_PAID_PRIOR",
    "AVG_PAYMENT_DELAY_PRIOR",
    "MAX_PAYMENT_DELAY_PRIOR",
    "PAYMENT_SUCCESS_RATE_PRIOR",

    "TOTAL_CLAIMS_PRIOR",
    "TOTAL_CLAIM_AMOUNT_PRIOR",
    "AVG_CLAIM_AMOUNT_PRIOR",
    "MAX_CLAIM_AMOUNT_PRIOR",
    "DAYS_SINCE_LAST_CLAIM",

    "HAS_PAYMENT_HISTORY_PRIOR",
    "HAS_CLAIM_HISTORY_PRIOR",
    "HAS_UNPAID_PAYMENT_AT_RENEWAL",
    "HAS_UNDERWRITING_RECORD",
    "PREMIUM_INCREASED_FLAG",
    "HIGH_PREMIUM_INCREASE_FLAG"
]

In [59]:
categorical_features = [
    "GENDER",
    "MARITAL_STATUS",
    "OCCUPATION",
    "STATE",
    "CUSTOMER_RISK_SEGMENT",

    "POLICY_TYPE",
    "SALES_CHANNEL",
    "PAYMENT_MODE",
    "RISK_BAND",

    "LIFESTYLE",
    "MEDICAL_HISTORY_FLAG",
    "SMOKER_FLAG",
    "OCCUPATION_RISK",
    "UW_DECISION",

    "CONTACT_CHANNEL"
]

Check Missing Values in Features

In [60]:
feature_check = pd.DataFrame({
    "COLUMN": numeric_features + categorical_features
})

In [61]:
feature_check["MISSING_COUNT"] = feature_check["COLUMN"].apply(lambda col: model_df[col].isna().sum())

In [62]:
feature_check["UNIQUE_VALUES"] = feature_check["COLUMN"].apply(lambda col: model_df[col].nunique())

Check Target vs Premium Increase Again

In [63]:
(
    model_df
    .groupby(
        "HIGH_PREMIUM_INCREASE_FLAG"
    )["NON_RENEWAL_FLAG"]
    .mean()
    .mul(100)
    .round(2)
)

HIGH_PREMIUM_INCREASE_FLAG
0     9.17
1    20.24
Name: NON_RENEWAL_FLAG, dtype: float64

In [64]:
model_df.groupby("HAS_CLAIM_HISTORY_PRIOR")["NON_RENEWAL_FLAG"].mean().mul(100).round(2)

HAS_CLAIM_HISTORY_PRIOR
0    9.75
1    9.71
Name: NON_RENEWAL_FLAG, dtype: float64

In [65]:
# Check Target vs Payment History

model_df[model_df["HAS_PAYMENT_HISTORY_PRIOR"] == 1].groupby("HAS_UNPAID_PAYMENT_AT_RENEWAL")["NON_RENEWAL_FLAG"].mean().mul(100).round(2)

HAS_UNPAID_PAYMENT_AT_RENEWAL
0    9.76
1    9.54
Name: NON_RENEWAL_FLAG, dtype: float64

In [ ]:
# One Important Synthetic-Data Limitation
# We earlier discovered serious date inconsistencies between:
# Policy periods
# Claims
# Premium transactions
# Renewal dates

# So even though we are using point-in-time filtering, this remains a synthetic-data project limitation.
# Document it.

In a real production project we'd require:
Validated event timestamps
Policy version history
As-of snapshots
Reliable renewal cutoff

In [66]:
MODEL_DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "curated"
)

MODEL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [67]:
model_df.to_csv(
    MODEL_DATA_DIR
    / "renewal_model_dataset.csv",
    index=False
)

In [68]:
model_check = pd.read_csv(MODEL_DATA_DIR / "renewal_model_dataset.csv", low_memory=False)

print(model_check.shape)
print(model_check["POLICY_ID"].duplicated().sum())
print(model_check["NON_RENEWAL_FLAG"].value_counts())

(19204, 60)
0
NON_RENEWAL_FLAG
0    17334
1     1870
Name: count, dtype: int64


Business prediction point
        ↓
Finalized outcomes only
        ↓
Positive class defined
        ↓
Leakage columns removed
        ↓
Static features selected
        ↓
Payments filtered to pre-renewal history
        ↓
Claims filtered to pre-renewal knowledge
        ↓
New business features engineered
        ↓
Missing-data meaning preserved
        ↓
Numeric/categorical feature sets defined
        ↓
Clean ML dataset created